# W-O-EN-01 — ABTS Activity Assay (ThermoFisher T-AOC kit, EEA023)

### SOP Header (controlled document)

| Field | Value |
|---|---|
| **Doc No** | SOP W-O-EN-01 |
| **Version** | 0.1 (DRAFT) |
| **Effective date** | _TBC_ |
| **Owner** | _TBC (iGEM 2026, DTU)_ |
| **Approved by** | _TBC_ |
| **Workflow ID** | W-O-EN-01 |
| **Workflow name** | ABTS colorimetric activity assay — reagent additions + develop |
| **Unit operations** | U-O-01 → U-O-02 → U-O-03 → U-C-01 |
| **Output** | One developed 96-well plate for A₄₁₄ readout, plus the generated, simulator-validated OT-2 `.py` protocol. |

> **Scope.** Samples and Trolox standards (10 µL each) are **pre-loaded** by the operator. This workflow adds the peroxidase application solution and the ABTS working solution, mixes, and stands 6 min before the off-deck 414 nm read (U-C-01). No heater-shaker is used.


## Steps (unit operations) — read in ~10 s

- **U-O-01** Add peroxidase application solution (20 µL) to each column
- **U-O-02** Add ABTS working solution (170 µL) to each column + mix
- **U-O-03** Stand 6 min at room temperature (colour development)
- **U-C-01** Read absorbance at 414 nm on the Clariostar (off-deck handoff)


## Run Record

In [1]:
run_record = {
    "run_id": "RUN-EN01-0000",        # PLACEHOLDER
    "operator": "",                    # PLACEHOLDER
    "datetime": "",                    # ISO 8601, filled at run time
    "notes_pre_run": "",
    "notes_during_run": "",
    "notes_deviations": "",
    "notes_post_run": "",
}
run_record


{'run_id': 'RUN-EN01-0000',
 'operator': '',
 'datetime': '',
 'notes_pre_run': '',
 'notes_during_run': '',
 'notes_deviations': '',
 'notes_post_run': ''}

## Parameters — single source of truth

Named variables with units; no magic numbers elsewhere. Placeholders marked. The
generated `.py` bakes these values in.


In [2]:
def render_params_block(params):
    """Render the ordered PARAMS list into a PARAMETERS block for the generated .py.
    PARAMS is the single source of truth; the generated file derives from it."""
    lines = ["# " + "=" * 74,
             "# PARAMETERS  (single source of truth - generated from the notebook)",
             "# " + "=" * 74]
    for name, value, comment in params:
        lines.append(f"{name} = {value!r}" + (f"  # {comment}" if comment else ""))
    return "\n".join(lines)

# --- API / hardware ---
API_LEVEL  = "2.16"
P300_MODEL = "p300_multi_gen2"
P20_MODEL  = "p20_multi_gen2"
P300_MOUNT = "left"
P20_MOUNT  = "right"

# --- Labware load names ---
ASSAY_PLATE_LOADNAME = "corning_96_wellplate_360ul_flat"   # clear flat-bottom for A414
RESERVOIR_LOADNAME   = "nest_12_reservoir_15ml"
TIPRACK_300_LOADNAME = "opentrons_96_tiprack_300ul"
TIPRACK_20_LOADNAME  = "opentrons_96_tiprack_20ul"

# --- Deck slots (no Heater-Shaker -> no adjacency constraints) ---
ASSAY_PLATE_SLOT = 2
TIPRACK_300_SLOT = 4
RESERVOIR_SLOT   = 5
TIPRACK_20_SLOT  = 6

# --- Plate / layout ---
NUM_SAMPLE_COLUMNS = 10    # PLACEHOLDER: columns of pre-loaded samples + standards (1-12)
PLATE_WELL_MAX_UL  = 360

# --- Reagent reservoir wells ---
PEROXIDASE_RESERVOIR_WELL = "A1"   # peroxidase application solution
ABTS_RESERVOIR_WELL       = "A2"   # ABTS working solution

# --- Volumes (per well) ---
SAMPLE_VOL_UL     = 10     # pre-loaded by operator (not pipetted; for volume check)
PEROXIDASE_VOL_UL = 20
ABTS_VOL_UL       = 170
TOTAL_VOL_PER_WELL_UL = SAMPLE_VOL_UL + PEROXIDASE_VOL_UL + ABTS_VOL_UL  # 200

# --- Mixing & development ---
MIX_REPS    = 3
MIX_VOL_UL  = 100          # < total well volume; P300 range
DEVELOP_MIN = 6            # kit: stand 6 min at RT before reading

# --- Tip strategy ---
# Peroxidase add: same reagent, no mixing -> reuse one tip set.
# ABTS add+mix touches well contents -> new tips per column to avoid carryover.
PEROXIDASE_NEW_TIP_PER_COLUMN = False
ABTS_NEW_TIP_PER_COLUMN       = True

# --- Output ---
OUTPUT_DIR     = "."
OUTPUT_PY_NAME = "W-O-EN-01_abts_activity_assay_ot2.py"
WORKFLOW_ID    = "W-O-EN-01"
PROTOCOL_NAME  = "W-O-EN-01 ABTS Activity Assay"
OWNER          = "iGEM 2026 DTU"
PROTOCOL_DESCRIPTION = ("ThermoFisher T-AOC ABTS reagent additions (peroxidase + ABTS "
                        "working solution) + mix + 6 min develop; samples pre-loaded; "
                        "read A414 off-deck.")

PARAMS = [
    ("P300_MODEL", P300_MODEL, ""),
    ("P20_MODEL", P20_MODEL, ""),
    ("P300_MOUNT", P300_MOUNT, ""),
    ("P20_MOUNT", P20_MOUNT, ""),
    ("ASSAY_PLATE_LOADNAME", ASSAY_PLATE_LOADNAME, "clear flat-bottom for A414"),
    ("RESERVOIR_LOADNAME", RESERVOIR_LOADNAME, ""),
    ("TIPRACK_300_LOADNAME", TIPRACK_300_LOADNAME, ""),
    ("TIPRACK_20_LOADNAME", TIPRACK_20_LOADNAME, ""),
    ("ASSAY_PLATE_SLOT", ASSAY_PLATE_SLOT, ""),
    ("TIPRACK_300_SLOT", TIPRACK_300_SLOT, ""),
    ("RESERVOIR_SLOT", RESERVOIR_SLOT, ""),
    ("TIPRACK_20_SLOT", TIPRACK_20_SLOT, ""),
    ("NUM_SAMPLE_COLUMNS", NUM_SAMPLE_COLUMNS, "PLACEHOLDER (1-12)"),
    ("PEROXIDASE_RESERVOIR_WELL", PEROXIDASE_RESERVOIR_WELL, ""),
    ("ABTS_RESERVOIR_WELL", ABTS_RESERVOIR_WELL, ""),
    ("PEROXIDASE_VOL_UL", PEROXIDASE_VOL_UL, ""),
    ("ABTS_VOL_UL", ABTS_VOL_UL, ""),
    ("MIX_REPS", MIX_REPS, ""),
    ("MIX_VOL_UL", MIX_VOL_UL, ""),
    ("DEVELOP_MIN", DEVELOP_MIN, ""),
    ("PEROXIDASE_NEW_TIP_PER_COLUMN", PEROXIDASE_NEW_TIP_PER_COLUMN, ""),
    ("ABTS_NEW_TIP_PER_COLUMN", ABTS_NEW_TIP_PER_COLUMN, ""),
]

DECK_SUMMARY = [
    f"slot {ASSAY_PLATE_SLOT}: assay plate ({ASSAY_PLATE_LOADNAME})",
    f"slot {TIPRACK_300_SLOT}: {TIPRACK_300_LOADNAME}",
    f"slot {RESERVOIR_SLOT}: {RESERVOIR_LOADNAME} (A1=peroxidase, A2=ABTS working)",
    f"slot {TIPRACK_20_SLOT}: {TIPRACK_20_LOADNAME}",
    "slot 12: fixed trash",
]
print("Parameters loaded. Total volume per well:", TOTAL_VOL_PER_WELL_UL, "uL")


Parameters loaded. Total volume per well: 200 uL


## Input validation (pre-flight)

In [3]:
assert 1 <= NUM_SAMPLE_COLUMNS <= 12, "NUM_SAMPLE_COLUMNS must be 1-12"
assert PEROXIDASE_VOL_UL > 0 and ABTS_VOL_UL > 0, "reagent volumes must be > 0"
assert TOTAL_VOL_PER_WELL_UL <= PLATE_WELL_MAX_UL, "well overfilled"
assert PEROXIDASE_RESERVOIR_WELL != ABTS_RESERVOIR_WELL, "reagents need distinct troughs"
assert 1 <= PEROXIDASE_VOL_UL <= 20, "peroxidase add expected on P20 (<=20 uL)"
assert 20 <= ABTS_VOL_UL <= 300, "ABTS add expected on P300 (20-300 uL)"
assert MIX_VOL_UL < TOTAL_VOL_PER_WELL_UL, "mix volume must be below well volume"
print("Input validation: PASS")


Input validation: PASS


## U-O-01 — Add peroxidase application solution

- **Goal:** Dispense 20 µL peroxidase application solution into each pre-loaded column.
- **Inputs:** Assay plate (10 µL sample/standard pre-loaded); reservoir `A1`; P20 multi.
- **Outputs:** Each well at sample + `PEROXIDASE_VOL_UL`.
- **Acceptance criteria:** `PEROXIDASE_VOL_UL` ≤ 20 (P20 range).
- **Record:** Peroxidase application-solution prep time (fresh 9:1 buffer:peroxidase), tip count.
- **Deviation handling:** Trough short → top up, resume.
- **Notes:** Same reagent, no mixing → one tip set reused.


In [4]:
def u_o_01_lines():
    return """
    # ---- U-O-01: Add peroxidase application solution ----
    p20.transfer(
        PEROXIDASE_VOL_UL,
        reservoir[PEROXIDASE_RESERVOIR_WELL],
        sample_cols,
        new_tip=("always" if PEROXIDASE_NEW_TIP_PER_COLUMN else "once"),
    )
"""
print("U-O-01 emitter ready")

U-O-01 emitter ready


## U-O-02 — Add ABTS working solution + mix

- **Goal:** Dispense 170 µL ABTS working solution into each column and mix to start the reaction.
- **Inputs:** Reservoir `A2` = ABTS working solution; P300 multi.
- **Outputs:** Each well at `TOTAL_VOL_PER_WELL_UL`, mixed.
- **Acceptance criteria:** ABTS working solution **< 30 min** old (kit stability); `ABTS_VOL_UL` ≤ 300.
- **Record:** ABTS working-solution prep clock time; mix reps/volume.
- **Deviation handling:** If > 30 min since prep, discard and re-make before dispensing.
- **Notes:** Mixing contacts well contents → **new tips per column** (`ABTS_NEW_TIP_PER_COLUMN=True`) to prevent column-to-column carryover. The reaction starts on contact — keep additions brisk and consistent across columns.


In [5]:
def u_o_02_lines():
    return """
    # ---- U-O-02: Add ABTS working solution + mix ----
    p300.transfer(
        ABTS_VOL_UL,
        reservoir[ABTS_RESERVOIR_WELL],
        sample_cols,
        new_tip=("always" if ABTS_NEW_TIP_PER_COLUMN else "once"),
        mix_after=(MIX_REPS, MIX_VOL_UL),
    )
"""
print("U-O-02 emitter ready")

U-O-02 emitter ready


## U-O-03 — Stand 6 min (colour development)

- **Goal:** Allow the ABTS⁺ colour to develop at room temperature.
- **Inputs:** Filled, mixed plate.
- **Outputs:** Developed plate ready for A₄₁₄.
- **Acceptance criteria:** `DEVELOP_MIN` elapsed for all wells; read promptly after.
- **Record:** Develop start time (t₀) for the plate.
- **Deviation handling:** Reader busy → record extra delay; colour drifts with time.
- **Notes:** Endpoint kinetics — consistent t₀-to-read across plates matters more than the absolute 6 min.


In [6]:
def u_o_03_lines():
    return """
    # ---- U-O-03: Stand 6 min (develop) ----
    ctx.delay(minutes=DEVELOP_MIN)
"""
print("U-O-03 emitter ready")

U-O-03 emitter ready


## U-C-01 — Read absorbance at 414 nm (Clariostar)

- **Goal:** Quantify ABTS⁺ at 414 nm.
- **Inputs:** Developed plate.
- **Outputs:** A₄₁₄ values → Trolox standard curve → activity.
- **Acceptance criteria:** Read within ~6 min window; standard curve R² acceptable.
- **Record:** Clariostar method file; read time vs. t₀.
- **Deviation handling:** Reader unavailable → read ASAP, note delay.
- **Notes:** Off-deck on OT-2; the protocol pauses for the operator to transfer the plate.


In [7]:
def u_c_01_lines():
    return """
    # ---- U-C-01: Hand off to Clariostar (414 nm) ----
    ctx.pause("U-C-01: Remove the plate and read A414 on the Clariostar.")
"""
print("U-C-01 emitter ready")

UNIT_OP_EMITTERS = [u_o_01_lines, u_o_02_lines, u_o_03_lines, u_c_01_lines]

U-C-01 emitter ready


## Assemble & generate the OT-2 protocol

In [8]:
def setup_lines():
    """Deck + labware + instruments (no Heater-Shaker)."""
    return f"""
    # ---- Deck, labware, instruments ----
    assay_plate = ctx.load_labware(ASSAY_PLATE_LOADNAME, ASSAY_PLATE_SLOT)
    tiprack_300 = ctx.load_labware(TIPRACK_300_LOADNAME, TIPRACK_300_SLOT)
    reservoir   = ctx.load_labware(RESERVOIR_LOADNAME, RESERVOIR_SLOT)
    tiprack_20  = ctx.load_labware(TIPRACK_20_LOADNAME, TIPRACK_20_SLOT)
    p300 = ctx.load_instrument(P300_MODEL, P300_MOUNT, tip_racks=[tiprack_300])
    p20  = ctx.load_instrument(P20_MODEL,  P20_MOUNT,  tip_racks=[tiprack_20])

    sample_cols = [assay_plate[f"A{{c}}"] for c in range(1, NUM_SAMPLE_COLUMNS + 1)]
"""

# Assemble the standalone OT-2 protocol from the unit-operation emitters.
import os, textwrap

_metadata = {
    "protocolName": PROTOCOL_NAME,
    "author": OWNER,
    "source": "Generated by " + WORKFLOW_ID + " literate notebook",
    "description": PROTOCOL_DESCRIPTION,
}

_header = (
    "# " + "=" * 74 + "\n"
    "# " + WORKFLOW_ID + "  |  " + PROTOCOL_NAME + "\n"
    "# Auto-generated from the literate notebook - edit PARAMETERS in the\n"
    "# notebook and re-generate; do not hand-edit this file.\n"
    "# " + "=" * 74 + "\n\n"
    "from opentrons import protocol_api\n\n"
    "metadata = " + repr(_metadata) + "\n"
    'requirements = {"robotType": "OT-2", "apiLevel": ' + repr(API_LEVEL) + "}\n\n"
)

_params_block = render_params_block(PARAMS) + "\n\n"

_run_open = "def run(ctx: protocol_api.ProtocolContext):\n"

# Order of unit operations inside run():
_body = setup_lines()
for _emit in UNIT_OP_EMITTERS:
    _body += _emit()

PROTOCOL_SRC = _header + _params_block + _run_open + _body

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PY_PATH = os.path.join(OUTPUT_DIR, OUTPUT_PY_NAME)
with open(OUTPUT_PY_PATH, "w") as _f:
    _f.write(PROTOCOL_SRC)
print("Generated:", OUTPUT_PY_PATH, f"({len(PROTOCOL_SRC)} chars)")


Generated: ./W-O-EN-01_abts_activity_assay_ot2.py (2988 chars)


## Validation gates (output checks + py_compile + opentrons_simulate)

In [9]:
# ---- Validation gates (mandatory before release) ----
import py_compile, subprocess, sys

results = {}

# 1) Output validation: destination wells unique, counts correct, mapping sane.
dest_wells = [f"{row}{col}" for col in range(1, NUM_SAMPLE_COLUMNS + 1)
              for row in "ABCDEFGH"]
results["unique_wells"] = (len(dest_wells) == len(set(dest_wells)))
results["well_count"] = (len(dest_wells) == NUM_SAMPLE_COLUMNS * 8)
results["columns_in_range"] = (1 <= NUM_SAMPLE_COLUMNS <= 12)
results["volume_fits_well"] = (TOTAL_VOL_PER_WELL_UL <= PLATE_WELL_MAX_UL)

# 2) Script validation: py_compile the generated protocol.
try:
    py_compile.compile(OUTPUT_PY_PATH, doraise=True)
    results["py_compile"] = True
except py_compile.PyCompileError as e:
    results["py_compile"] = False
    print(e)

# 3) Simulation validation: opentrons_simulate the generated protocol.
proc = subprocess.run(["opentrons_simulate", OUTPUT_PY_PATH],
                      capture_output=True, text=True)
results["opentrons_simulate"] = (proc.returncode == 0)
if proc.returncode != 0:
    print(proc.stdout[-2000:])
    print(proc.stderr[-2000:])

print("\nValidation results:")
for k, v in results.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")

all_pass = all(results.values())
print("\nOVERALL:", "PASS - release OK" if all_pass else "FAIL - do not release")
assert all_pass, "Validation gate failed - see above."



Validation results:
  PASS  unique_wells
  PASS  well_count
  PASS  columns_in_range
  PASS  volume_fits_well
  PASS  py_compile
  PASS  opentrons_simulate

OVERALL: PASS - release OK


## Artifacts & run summary

In [10]:
# ---- Artifacts & run summary ----
print("ARTIFACT (generated OT-2 protocol):")
print("  ", OUTPUT_PY_PATH)
print("\nKey parameters used:")
for name, value, _ in PARAMS:
    print(f"  {name} = {value!r}")
print("\nDeck summary:")
for line in DECK_SUMMARY:
    print("  ", line)


ARTIFACT (generated OT-2 protocol):
   ./W-O-EN-01_abts_activity_assay_ot2.py

Key parameters used:
  P300_MODEL = 'p300_multi_gen2'
  P20_MODEL = 'p20_multi_gen2'
  P300_MOUNT = 'left'
  P20_MOUNT = 'right'
  ASSAY_PLATE_LOADNAME = 'corning_96_wellplate_360ul_flat'
  RESERVOIR_LOADNAME = 'nest_12_reservoir_15ml'
  TIPRACK_300_LOADNAME = 'opentrons_96_tiprack_300ul'
  TIPRACK_20_LOADNAME = 'opentrons_96_tiprack_20ul'
  ASSAY_PLATE_SLOT = 2
  TIPRACK_300_SLOT = 4
  RESERVOIR_SLOT = 5
  TIPRACK_20_SLOT = 6
  NUM_SAMPLE_COLUMNS = 10
  PEROXIDASE_RESERVOIR_WELL = 'A1'
  ABTS_RESERVOIR_WELL = 'A2'
  PEROXIDASE_VOL_UL = 20
  ABTS_VOL_UL = 170
  MIX_REPS = 3
  MIX_VOL_UL = 100
  DEVELOP_MIN = 6
  PEROXIDASE_NEW_TIP_PER_COLUMN = False
  ABTS_NEW_TIP_PER_COLUMN = True

Deck summary:
   slot 2: assay plate (corning_96_wellplate_360ul_flat)
   slot 4: opentrons_96_tiprack_300ul
   slot 5: nest_12_reservoir_15ml (A1=peroxidase, A2=ABTS working)
   slot 6: opentrons_96_tiprack_20ul
   slot 12: fixe

## Change Log

| Date | Version | Author | Summary of changes |
|---|---|---|---|
| _TBC_ | 0.1 | Claude (draft) | Initial literate notebook generating + simulating the OT-2 ABTS activity-assay protocol. |
